# Capítulo 26 · Laboratório 26: Python para o analista técnico
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (40 ações desde 2005). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap26.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. Os números ilustram armadilhas de programação: dado sujo, aquecimento, Bollinger com desvio da amostra, semanal errado, a linha do `shift` esquecida e o IFR com média simples. A conferência com a TA-Lib é opcional: no Colab, rode `!pip install -q TA-Lib` antes (se não instalar, a Parte 2 é pulada). O `yfinance` é uma interface não oficial do Yahoo Finance; a cópia local guarda os dados para que os números não mudem entre as suas execuções.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import re
import os
import time
import numpy as np
import pandas as pd
import yfinance as yf

B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
INICIO, FIM = "2005-01-01", "2026-10-01"      # datas fixas: dados até 30/09/2026
CACHE = "dados_cap26.pkl"
CUSTO = 0.001


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


# indicadores escritos à mão (usados em todas as partes)
def mms(c, n):
    return c.rolling(n).mean()


def mme(c, n):
    return c.ewm(span=n, adjust=False).mean()


def ifr(c, n=14):
    """IFR de Wilder: médias das altas e das baixas suavizadas por 1/n (não é média simples)."""
    d = c.diff()
    alta = d.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
    baixa = (-d.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean()
    return 100 - 100 / (1 + alta / baixa)


def ifr_media_simples(c, n=14):
    """Versão comum em códigos prontos: média simples no lugar da suavização de Wilder."""
    d = c.diff()
    return 100 - 100 / (1 + d.clip(lower=0).rolling(n).mean() / (-d.clip(upper=0)).rolling(n).mean())


def atr(h, l, c, n=14):
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def macd(c, rapida=12, lenta=26, sinal=9):
    linha = mme(c, rapida) - mme(c, lenta)
    return linha, mme(linha, sinal)


def bollinger(c, n=20, k=2, ddof=0):
    m, s = c.rolling(n).mean(), c.rolling(n).std(ddof=ddof)
    return m - k * s, m, m + k * s

## Parte 1: dados

In [ ]:
if os.path.exists(CACHE):
    dados, brutos, cal = pd.read_pickle(CACHE)
else:
    dados = yf.download(TICKERS, start=INICIO, end=FIM, auto_adjust=True, progress=False, group_by="ticker")
    brutos = yf.download(TICKERS, start=INICIO, end=FIM, auto_adjust=False, progress=False, group_by="ticker")
    cal = {True: yf.download("^BVSP", start=INICIO, end=FIM, progress=False, multi_level_index=False).index,
           False: yf.download("^GSPC", start=INICIO, end=FIM, progress=False, multi_level_index=False).index}
    confere({"as 40 ações (ajustadas)": dados, "as 40 ações (sem ajuste)": brutos,
             "^BVSP": cal[True], "^GSPC": cal[False]})
    pd.to_pickle((dados, brutos, cal), CACHE)

prob = {"linhas vazias (B3 e EUA na mesma tabela)": 0, "preço zero ou negativo": 0, "máxima abaixo da mínima": 0,
        "fechamento fora da barra": 0, "volume zero (B3)": 0, "volume zero (EUA)": 0, "variação acima de 25% em um dia": 0,
        "pregão do índice sem dado da ação": 0}
limpos = {}
for t in TICKERS:
    d = dados[t]
    prob["linhas vazias (B3 e EUA na mesma tabela)"] += int(d["Close"].isna().sum())
    d = d.dropna()
    prob["preço zero ou negativo"] += int((d[["Open", "High", "Low", "Close"]] <= 0).any(axis=1).sum())
    prob["máxima abaixo da mínima"] += int((d["High"] < d["Low"]).sum())
    prob["fechamento fora da barra"] += int(((d["Close"] > d["High"] * 1.0001) | (d["Close"] < d["Low"] * 0.9999)).sum())
    prob["volume zero (B3)" if t.endswith(".SA") else "volume zero (EUA)"] += int((d["Volume"] == 0).sum())
    prob["variação acima de 25% em um dia"] += int((d["Close"].pct_change().abs() > 0.25).sum())
    idx = cal[t.endswith(".SA")]
    prob["pregão do índice sem dado da ação"] += int(len(idx[(idx >= d.index[0]) & (idx <= d.index[-1])].difference(d.index)))
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] >= d["Low"])
    limpos[t] = d[ok]
mostra("Parte 1. O que veio errado do provedor (40 ações, 2005 a 30/09/2026)")
mostra(f"   linhas no total: {sum(len(dados[t].dropna()) for t in TICKERS):_}")
for k, v in prob.items():
    mostra(f"   {k:<46} {v:>6_}")
z = (limpos["SUZB3.SA"]["Volume"] == 0)
parado = limpos["SUZB3.SA"].loc[:"2017-11-09"]
mostra(f"   exemplo: SUZB3 tem {int(z.sum()):_} dias com volume zero; até 09/11/2017, {int((parado['High'] == parado['Low']).sum()):_}"
       f" pregões com máxima igual à mínima (preço parado: as ações negociadas eram as preferenciais SUZB5 e SUZB6)")
limpos["SUZB3.SA"] = limpos["SUZB3.SA"].loc["2017-11-10":]      # a SUZB3 negocia de verdade a partir de 10/11/2017

mostra("\nPreço ajustado (proventos e desdobramentos) x preço da tela, retorno total 2010 a 30/09/2026")
for t in ["PETR4.SA", "ITUB4.SA", "KO", "AAPL"]:
    a = limpos[t]["Close"].loc["2010":]
    b = brutos[t]["Close"].dropna().loc["2010":]
    mostra(f"   {t[:-3] if t.endswith('.SA') else t:<6} ajustado {a.iloc[-1] / a.iloc[0] - 1:>+8_.0%} | da tela {b.iloc[-1] / b.iloc[0] - 1:>+8_.0%}")

## Parte 2: conferência contra a TA-Lib

In [ ]:
try:
    import talib
    mostra("\nParte 2. Indicadores à mão x TA-Lib (maior diferença absoluta depois das 300 primeiras barras, 40 ações)")
    dif = {"MMS(20)": 0, "MME(20)": 0, "IFR(14)": 0, "ATR(14)": 0, "MACD(12,26)": 0, "Banda superior (20,2)": 0}
    for t in TICKERS:
        d = limpos[t]
        c, h, l = d["Close"], d["High"], d["Low"]
        C, H, L = c.to_numpy(float), h.to_numpy(float), l.to_numpy(float)
        par = {"MMS(20)": (mms(c, 20), talib.SMA(C, 20)), "MME(20)": (mme(c, 20), talib.EMA(C, 20)),
               "IFR(14)": (ifr(c), talib.RSI(C, 14)), "ATR(14)": (atr(h, l, c), talib.ATR(H, L, C, 14)),
               "MACD(12,26)": (macd(c)[0], talib.MACD(C)[0]),
               "Banda superior (20,2)": (bollinger(c)[2], talib.BBANDS(C, 20, 2, 2)[0])}
        for k, (a, b) in par.items():
            rel = (np.abs(a.to_numpy()[300:] - b[300:]) / np.abs(c.to_numpy()[300:])).max() if k != "IFR(14)" \
                else np.abs(a.to_numpy()[300:] - b[300:]).max()
            dif[k] = max(dif[k], rel)
    for k, v in dif.items():
        mostra(f"   {k:<22} {v:.2e} {'pontos de IFR' if k == 'IFR(14)' else 'do preço'}")
except ImportError:
    mostra("\nParte 2: TA-Lib não instalada (pip install TA-Lib); conferência pulada")

## Parte 3: aquecimento

In [ ]:
mostra("\nParte 3. Aquecimento: diferença para a série com todo o histórico, conforme as barras de aquecimento")
mostra("   barras antes | IFR(14), maior diferença em pontos | MME(200), maior diferença em % do preço")
alvo_ini = pd.Timestamp("2024-01-02")
for aq in [14, 30, 60, 100, 200, 400, 800]:
    m_ifr, m_mme = 0.0, 0.0
    for t in TICKERS:
        c = limpos[t]["Close"]
        i0 = c.index.searchsorted(alvo_ini)
        cheio_i, cheio_m = ifr(c).iloc[i0:], mme(c, 200).iloc[i0:]
        curto = c.iloc[max(i0 - aq, 0):]
        m_ifr = max(m_ifr, (ifr(curto).loc[cheio_i.index] - cheio_i).abs().max())
        m_mme = max(m_mme, ((mme(curto, 200).loc[cheio_m.index] - cheio_m).abs() / c.loc[cheio_m.index]).max())
    mostra(f"   {aq:>12} | {m_ifr:>10.2f} | {m_mme:>10.2%}")

## Parte 4: Bollinger, amostra x população

In [ ]:
n_pop = n_amo = 0
for t in TICKERS:
    c = limpos[t]["Close"]
    acima = c > mms(c, 200)
    n_pop += int((acima & (c < bollinger(c, ddof=0)[0])).sum())
    n_amo += int((acima & (c < bollinger(c, ddof=1)[0])).sum())
mostra(f"\nParte 4. Bollinger com desvio da população (ddof=0) x da amostra (ddof=1, o padrão do pandas)")
mostra(f"   bandas da amostra são {np.sqrt(20 / 19) - 1:.1%} mais largas | sinais de %b < 0 acima da MM200: "
      f"população {n_pop:_} x amostra {n_amo:_} ({n_amo / n_pop - 1:+.0%})")

## Parte 5: diário para semanal

In [ ]:
certo_regra = {"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"}
sem_total = sem_max_errada = sem_min_errada = 0
for t in TICKERS:
    d = limpos[t]
    certo = d.resample("W-FRI").agg(certo_regra).dropna()
    errado = d.resample("W-FRI").last().dropna()            # pega a última linha da semana para tudo
    sem_total += len(certo)
    sem_max_errada += int((errado["High"] != certo["High"]).sum())
    sem_min_errada += int((errado["Low"] != certo["Low"]).sum())
mostra(f"\nParte 5. Semanal feito com .last() em tudo: máxima errada em {sem_max_errada / sem_total:.0%} das semanas,"
      f" mínima errada em {sem_min_errada / sem_total:.0%} ({sem_total:_} semanas)")

## Parte 6: olhar o futuro e conferência vetorizado x laço

In [ ]:
def teste_vetorizado(c, futuro=False):
    """Comprado quando a MME de 9 está acima da de 21; a posição de hoje vem do sinal de ontem (shift)."""
    sinal = (mme(c, 9) > mme(c, 21)).astype(int)
    pos = sinal if futuro else sinal.shift(1).fillna(0)
    ret = c.pct_change().fillna(0)
    trocas = pos.diff().abs().fillna(0)
    return pos * ret - trocas * CUSTO / 2


def teste_laco(c):
    m9, m21, C = mme(c, 9).to_numpy(), mme(c, 21).to_numpy(), c.to_numpy()
    out, pos = np.zeros(len(C)), 0
    for i in range(1, len(C)):
        novo = 1 if m9[i - 1] > m21[i - 1] else 0                 # decide com o que se sabia ontem
        if novo != pos:
            out[i] -= CUSTO / 2                                   # meia volta de custo a cada troca
        pos = novo
        out[i] += pos * (C[i] / C[i - 1] - 1)
    return pd.Series(out, index=c.index)


mostra("\nParte 6. Cruzamento de MME 9 e 21, comprado ou fora, com custo (média das 40 ações, 2005 a 2026)")
ok, t_vet, t_laco, r_ok, r_fut, r_bh = 0, 0.0, 0.0, [], [], []
for t in TICKERS:
    c = limpos[t]["Close"]
    anos = len(c) / 252
    t0 = time.perf_counter(); v = teste_vetorizado(c); t_vet += time.perf_counter() - t0
    t0 = time.perf_counter(); lc = teste_laco(c); t_laco += time.perf_counter() - t0
    ok += int(np.allclose(v.to_numpy(), lc.to_numpy(), atol=1e-12))
    r_ok.append((1 + v).prod() ** (1 / anos) - 1)
    r_fut.append((1 + teste_vetorizado(c, futuro=True)).prod() ** (1 / anos) - 1)
    r_bh.append((c.iloc[-1] / c.iloc[0]) ** (1 / anos) - 1)
mostra(f"   com shift (correto)        {np.mean(r_ok):+.1%} ao ano")
mostra(f"   sem shift (olhou o futuro) {np.mean(r_fut):+.1%} ao ano")
mostra(f"   comprar e segurar          {np.mean(r_bh):+.1%} ao ano")
mostra(f"   vetorizado e laço iguais em {ok} de {len(TICKERS)} ações | tempo: vetorizado {t_vet:.2f} s, laço {t_laco:.2f} s")

In [ ]:
import matplotlib.pyplot as plt
c = limpos["PETR4.SA"]["Close"]
sinal = (mme(c, 9) > mme(c, 21)).astype(int)
ret = c.pct_change().fillna(0)
for pos, est, rot in [(sinal.shift(1).fillna(0), "-", "com shift"), (sinal, "--", "sem shift (olhou o futuro)")]:
    curva = (1 + pos * ret - pos.diff().abs().fillna(0) * CUSTO / 2).cumprod()
    plt.plot(curva.index, curva, color="black", ls=est, label=rot)
plt.plot(c.index, c / c.iloc[0], color="gray", ls=":", label="comprar e segurar")
plt.yscale("log")
plt.title("PETR4: R$ 1 no cruzamento de médias de 9 e 21")
plt.legend()
plt.show()

## Parte 7: IFR de Wilder x IFR com média simples

In [ ]:
corr, s_w, s_s, r_w, r_s = [], 0, 0, [], []
for t in TICKERS:
    c = limpos[t]["Close"]
    a, b = ifr(c, 14), ifr_media_simples(c, 14)
    corr.append(a.corr(b))
    s_w += int(((a < 30) & (a.shift() >= 30)).sum())
    s_s += int(((b < 30) & (b.shift() >= 30)).sum())
    acima = (c > mms(c, 200)).to_numpy()
    C = c.to_numpy()
    sai = (c > mms(c, 5)).to_numpy()
    for serie, dest in ((ifr(c, 2).to_numpy(), r_w), (ifr_media_simples(c, 2).to_numpy(), r_s)):
        for i in range(201, len(C) - 11):
            if acima[i] and serie[i] < 10:
                k = next((k for k in range(i + 1, i + 11) if sai[k]), i + 10)
                dest.append(C[k] / C[i] - 1 - CUSTO)
mostra(f"\nParte 7. Mesmo nome, outro indicador: IFR(14) de Wilder x com média simples")
mostra(f"   correlação média {np.mean(corr):.2f} | cruzamentos para baixo de 30: Wilder {s_w:_}, média simples {s_s:_}")
mostra(f"   IFR(2) de Connors: Wilder {len(r_w):_} operações, {np.mean(r_w):+.2%} com custo, acerto {np.mean(np.array(r_w) > 0):.0%}"
      f" | média simples {len(r_s):_} operações, {np.mean(r_s):+.2%}, acerto {np.mean(np.array(r_s) > 0):.0%}")

## Parte 8: scanner

In [ ]:
linhas = []
for t in [x + ".SA" for x in B3]:
    d = limpos[t]
    c = d["Close"]
    linhas.append({"ativo": t[:-3], "fech.": round(c.iloc[-1], 2),
                   "acima MM200": bool(c.iloc[-1] > mms(c, 200).iloc[-1]),
                   "IFR(2)": round(ifr(c, 2).iloc[-1], 1),
                   "máx. de 52 sem.": bool(d["High"].iloc[-1] >= d["High"].iloc[-252:].max()),
                   "dist. MM50": f"{c.iloc[-1] / mms(c, 50).iloc[-1] - 1:+.1%}"})
S = pd.DataFrame(linhas).set_index("ativo")
mostra(f"\nParte 8. Scanner das 20 ações da B3 em {limpos['PETR4.SA'].index[-1]:%d/%m/%Y}")
mostra(f"   acima da MM200: {S['acima MM200'].mean():.0%} | IFR(2) < 10 e acima da MM200: "
      f"{', '.join(S[(S['IFR(2)'] < 10) & S['acima MM200']].index) or 'nenhuma'} | na máxima de 52 semanas: "
      f"{', '.join(S[S['máx. de 52 sem.']].index) or 'nenhuma'}")
mostra(S.sort_values("IFR(2)").head(8).to_string())

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, as funções `ifr` e `ifr_media_simples` da célula de abertura. Mostre a única troca entre elas (`ewm(alpha=1 / n, adjust=False)` contra `rolling(n).mean()`) e por que essa troca muda o indicador, e não só o número." Confira com a Parte 7, que imprime a correlação média de 0,87 entre os dois e os cruzamentos para baixo de 30 (Wilder 2.152, média simples 5.401).
2. **Adaptar:** "Acrescente [ATIVO] ao scanner da Parte 8. Baixe o ativo à parte, com o ticker do Yahoo Finance, as mesmas datas (`INICIO` e `FIM`) e a mesma limpeza da Parte 1, e mostre a linha dele com as mesmas colunas, abaixo da tabela, sem mudar a conta das 20 ações da B3. Não mude as médias de 50 e 200 nem o IFR(2)." Antes de olhar o ativo novo, confira se a Parte 8 continua mostrando 65% das 20 ações acima da MM200 em 30/09/2026 e a SUZB3 com IFR(2) de 3,7, como na Tabela 26.4.

Mais pedidos deste capítulo (entender e aplicar ao seu caso) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 1, inclua a regra de Kaufman: variação acima de 4% num dia. Quantos casos aparecem? Quantos são crises reais?
2. Na Parte 3, troque o IFR(14) pelo IFR(2). Quantas barras de aquecimento ele precisa?
3. Na Parte 6, troque as médias de 9 e 21 por 50 e 200. A diferença entre com e sem shift diminui? Por quê?
4. Na Parte 8, inclua as 20 ações dos EUA e uma coluna com o ATR em % do preço.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.